In [2]:
# Install all packages required by this notebook.
# requirements-gpu.txt pins TensorFlow 2.10.1 because native Windows GPU
# support was dropped in TensorFlow 2.11.
# %pip install -q -r requirements-gpu.txt


In [3]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("ajaysonicu/deepfakefusion-399k-realfake-faces")

print("Path to dataset files:", path)

100%|██████████| 6.67G/6.67G [00:42<00:00, 168MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ajaysonicu/deepfakefusion-399k-realfake-faces/versions/1


In [4]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Activation, Dropout, BatchNormalization, GlobalAveragePooling2D

SEED = 42
# Seeds Python's `random`, NumPy and TensorFlow in one call
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

if gpus:
    print("GPUs:", gpus)
else:
    print(
        "No GPU detected. This notebook will run on CPU; verify the "
        "Python 3.10 environment, NVIDIA driver, and TensorFlow 2.10.1."
    )

TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
from pathlib import Path

dataset_root = Path(path) / "deepfakefusion_v2_sixsource_dataset"

print("Dataset root:", dataset_root)
print("\nDirectory structure:")
for split in sorted(dataset_root.iterdir()):
    if split.is_dir():
        print(f"[DIR] {split.name}")
        for class_dir in sorted(split.iterdir()):
            if class_dir.is_dir():
                print(f"    [DIR] {class_dir.name}")

Dataset root: /root/.cache/kagglehub/datasets/ajaysonicu/deepfakefusion-399k-realfake-faces/versions/1/deepfakefusion_v2_sixsource_dataset

Directory structure:
[DIR] metadata
[DIR] test
    [DIR] fake
    [DIR] real
[DIR] train
    [DIR] fake
    [DIR] real
[DIR] val
    [DIR] fake
    [DIR] real


In [6]:
from pathlib import Path

from PIL import Image

#extracting FFT features from images to use as input for the model
#https://emerginginvestigators.org/articles/24-270

def spectrum_analysis_FFT(image, image_size=(128, 128)):
    """Return a centered log-magnitude 2D FFT for one RGB image."""
    if isinstance(image, (str, Path)):
        image = Image.open(image).convert("RGB").resize(image_size)
        image = np.asarray(image, dtype=np.float32) / 255.0
    else:
        image = np.asarray(image, dtype=np.float32)
        if image.max() > 1.0:
            image = image / 255.0
        if image.ndim != 3 or image.shape[-1] != 3:
            raise ValueError("Expected an RGB image with shape (height, width, 3).")
        image = np.asarray(
            Image.fromarray((image * 255).clip(0, 255).astype(np.uint8)).resize(image_size),
            dtype=np.float32,
        ) / 255.0

    luminance = (
        0.299 * image[:, :, 0]
        + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]
    )
    luminance -= luminance.mean()
    window = np.outer(np.hanning(luminance.shape[0]), np.hanning(luminance.shape[1]))
    fft_shifted = np.fft.fftshift(np.fft.fft2(luminance * window))
    return np.log1p(np.abs(fft_shifted))


def _frequency_masks(shape):
    """Create center-relative low, middle, and high frequency masks."""
    height, width = shape
    frequencies_y = np.fft.fftshift(np.fft.fftfreq(height))
    frequencies_x = np.fft.fftshift(np.fft.fftfreq(width))
    frequency_x, frequency_y = np.meshgrid(frequencies_x, frequencies_y)
    radius = np.sqrt(frequency_x**2 + frequency_y**2)
    return {
        "low": radius < 0.125,
        "mid": (radius >= 0.125) & (radius < 0.30),
        "high": radius >= 0.30,
    }


def extract_FFT_param(image_path, label, class_name, image_size=(128, 128)):
    """Extract labeled FFT statistics for one image."""
    magnitude_spectrum = spectrum_analysis_FFT(image_path, image_size)
    masks = _frequency_masks(magnitude_spectrum.shape)
    power = np.square(magnitude_spectrum)
    total_power = power.sum() + 1e-8
    features = {
        "path": str(image_path),
        "label": int(label),
        "class_name": class_name,
        "mean_magnitude": float(magnitude_spectrum.mean()),
        "std_magnitude": float(magnitude_spectrum.std()),
    }
    for band_name, mask in masks.items():
        features[f"{band_name}_frequency"] = float(magnitude_spectrum[mask].mean())
        features[f"{band_name}_energy_ratio"] = float(power[mask].sum() / total_power)
    features["high_to_low_energy_ratio"] = float(
        features["high_energy_ratio"] / (features["low_energy_ratio"] + 1e-8)
    )
    return features


def associate_FFT_features_with_images(
    dataset,
    max_per_class=None,
    image_size=(128, 128),
    seed=42,
):
    """Return one labeled FFT-feature dictionary per image in fake/real folders."""
    dataset = Path(dataset)
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    rng = np.random.default_rng(seed)
    features_list = []

    for label, class_name in enumerate(("fake", "real")):
        class_dir = dataset / class_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Expected class directory: {class_dir}")
        image_paths = sorted(
            path for path in class_dir.rglob("*")
            if path.is_file() and path.suffix.lower() in image_extensions
        )
        if max_per_class is not None and len(image_paths) > max_per_class:
            selected_indices = rng.choice(len(image_paths), size=max_per_class, replace=False)
            image_paths = [image_paths[index] for index in sorted(selected_indices)]
        for image_path in image_paths:
            features_list.append(
                extract_FFT_param(image_path, label, class_name, image_size)
            )
    return features_list

In [7]:
IMG_SIZE = (128, 128)
batch_size = 32

train_dir = dataset_root / "train"
val_dir = dataset_root / "val"
test_dir = dataset_root / "test"

# Class names come from the sub-folder names (alphabetical: fake=0, real=1),
# label_mode="binary" gives 0/1 float labels that match Dense(1, sigmoid).
train_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(train_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Shuffle validation too: with shuffle=False, taking the first N batches
# would only return images from the first class folder ("fake").
val_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(val_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)

# Test set stays unshuffled so predictions line up with test_ds.file_paths
test_full_ds = tf.keras.utils.image_dataset_from_directory(
    str(test_dir),
    image_size=IMG_SIZE,
    batch_size=batch_size,
    label_mode="binary",
    shuffle=False,
)

class_names = train_full_ds.class_names
print("Classes:", {name: i for i, name in enumerate(class_names)})
print("Training images:", len(train_full_ds.file_paths))
print("Validation images:", len(val_full_ds.file_paths))
print("Test images:", len(test_full_ds.file_paths))

Found 319994 files belonging to 2 classes.
Found 40000 files belonging to 2 classes.
Found 39998 files belonging to 2 classes.
Classes: {'fake': 0, 'real': 1}
Training images: 319994
Validation images: 40000
Test images: 39998


In [8]:
import pandas as pd

# Use a balanced, reproducible sample for exploratory FFT analysis.
train_fft_features = associate_FFT_features_with_images(
    train_dir,
    max_per_class=1000,
    image_size=IMG_SIZE,
    seed=SEED,
)
train_fft_df = pd.DataFrame(train_fft_features)

fft_summary = (
    train_fft_df.groupby("class_name")
    [[
        "low_frequency",
        "mid_frequency",
        "high_frequency",
        "low_energy_ratio",
        "mid_energy_ratio",
        "high_energy_ratio",
        "high_to_low_energy_ratio",
    ]]
    .agg(["count", "mean", "std", "median"])
)

display(fft_summary)
display(train_fft_df.head())

low_frequency                               mid_frequency  \
                   count      mean       std    median         count   
class_name                                                             
fake                1000  2.601847  0.285966  2.632515          1000   
real                1000  2.679159  0.261363  2.699078          1000   

                                         high_frequency            ...  \
                mean       std    median          count      mean  ...   
class_name                                                         ...   
fake        1.203755  0.313810  1.264030           1000  0.440545  ...   
real        1.281621  0.280449  1.319044           1000  0.472016  ...   

           mid_energy_ratio           high_energy_ratio                      \
                        std    median             count      mean       std   
class_name                                                                    
fake               0.049048  0.419032              1000  0.188934  0.092627   
real               0.043979  0.424069              1000  0.200423  0.089068   

                     high_to_low_energy_ratio                                
              median                    count      mean       std    median  
class_name                                                                   
fake        0.218429                     1000  0.567389  0.367975  0.616695  
real        0.224997                     1000  0.615217  0.362361  0.641544  

[2 rows x 28 columns]

,path,label,class_name,mean_magnitude,std_magnitude,low_frequency,low_energy_ratio,mid_frequency,mid_energy_ratio,high_frequency,high_energy_ratio,high_to_low_energy_ratio
0,/root/.cache/kagglehub/datasets/ajaysonicu/dee...,0,fake,0.946159,0.714687,2.863277,0.307918,1.486651,0.420557,0.639874,0.271525,0.881808
1,/root/.cache/kagglehub/datasets/ajaysonicu/dee...,0,fake,1.152111,0.808336,3.044352,0.244168,1.763470,0.417064,0.824316,0.338768,1.387434
2,/root/.cache/kagglehub/datasets/ajaysonicu/dee...,0,fake,0.306565,0.475141,1.798296,0.610789,0.572767,0.338605,0.118754,0.050607,0.082855
3,/root/.cache/kagglehub/datasets/ajaysonicu/dee...,0,fake,0.890923,0.688185,2.748951,0.318253,1.368346,0.409370,0.609266,0.272377,0.855851
4,/root/.cache/kagglehub/datasets/ajaysonicu/dee...,0,fake,0.753400,0.641237,2.600996,0.361986,1.202025,0.414773,0.481870,0.223240,0.616710


In [9]:
from IPython.display import clear_output

class LiveTrainingPlot(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.train_loss = []
        self.val_loss = []
        self.train_acc = []
        self.val_acc = []

        self.best_val_loss = float('inf')
        self.best_epoch = 0
        self.best_val_acc = 0

    def on_epoch_end(self, epoch, logs=None):

        # Current epoch metrics
        train_loss = logs['loss']
        val_loss = logs['val_loss']
        train_acc = logs['accuracy']
        val_acc = logs['val_accuracy']

        # Save epoch metrics
        self.train_loss.append(logs['loss'])
        self.val_loss.append(logs['val_loss'])
        self.train_acc.append(logs['accuracy'])
        self.val_acc.append(logs['val_accuracy'])

        # Check if this is the best epoch so far
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.best_val_acc = val_acc
            self.best_epoch = epoch + 1

        best_epoch_idx = self.best_epoch - 1

        # Clear previous plot
        clear_output(wait=True)
        epochs_completed = range(1, len(self.train_loss) + 1)
        plt.figure(figsize=(12, 5))

        # Loss curves
        plt.subplot(1, 2, 1)
        plt.plot(
            epochs_completed,
            self.train_loss,
            marker='o',
            label='Training Loss'
        )
        plt.plot(
            epochs_completed,
            self.val_loss,
            marker='o',
            label='Validation Loss'
        )
        plt.xlabel('Epoch')
        plt.ylabel('Binary Cross-Entropy Loss')
        plt.title('Training vs Validation Loss')
        plt.legend()
        plt.grid(True)
        # Accuracy curves
        plt.subplot(1, 2, 2)
        plt.plot(
            epochs_completed,
            self.train_acc,
            marker='o',
            label='Training Accuracy'
        )
        plt.plot(
            epochs_completed,
            self.val_acc,
            marker='o',
            label='Validation Accuracy'

        )
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.title('Training vs Validation Accuracy')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()



        print(f"Previous EPOCH: {epoch + 1}")
        print(f"Train Loss: {train_loss:.4f} | "f"Train Accuracy: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | "f"Val Accuracy:   {val_acc:.4f}")

        print(f"\nBEST EPOCH SO FAR: {self.best_epoch}")
        print(f"Train Loss: {self.train_loss[best_epoch_idx]:.4f} | "f"Train Accuracy: {self.train_acc[best_epoch_idx]:.4f}")
        print(f"Val Loss: {self.best_val_loss:.4f} | "f"Val Accuracy: {self.best_val_acc:.4f}")

    def on_train_end(self, logs=None):
        print(f"Training completed for {len(self.train_loss)} epochs")
        print(f"Best Epoch: {self.best_epoch}\n" f"Best Validation Loss: {self.best_val_loss:.4f}\n" f"Validation Accuracy at Best Epoch: {self.best_val_acc:.4f}")

        print("\nEarlyStopping restored the model weights " f"from Epoch {self.best_epoch}.")

In [10]:
%pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.7 MB/s eta 0:00:00


In [11]:
# Sampling the training data so the build process takes a reasonable amount of time
# resource: https://www.geeksforgeeks.org/machine-learning/python-image-classification-using-keras/

train_sample_size = 100000
validation_sample_size = 80000  # bumped up for a more reliable comparison between trials (was 2000)
# Keep search and validation budgets explicit so experiments finish predictably.
tuning_epochs = 20
validation_epochs = 20

input_shape = (*IMG_SIZE, 3)   # image_dataset_from_directory always yields channels-last

AUTOTUNE = tf.data.AUTOTUNE

import keras_tuner as kt
import numpy as np

# Rescale pixels to [0, 1]
normalize = lambda x, y: (x / 255.0, y)

# Random flips/rotation/contrast, only applied to training data, to help the model generalize
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomContrast(0.1),
])

# .take() counts batches, not images. The file list was shuffled once with SEED,
# so the first N batches are a random sample. .cache() then freezes that sample
# so every epoch sees the same images (~2 GB of RAM for the training subset;
# lower train_sample_size if that is too much).
train_ds = (train_full_ds
            .take(train_sample_size // batch_size)
            .map(normalize, num_parallel_calls=AUTOTUNE)
            .cache()
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .prefetch(AUTOTUNE))

val_ds = (val_full_ds
          .take(validation_sample_size // batch_size)
          .map(normalize, num_parallel_calls=AUTOTUNE)
          .cache()
          .prefetch(AUTOTUNE))

# Full test set, same preprocessing, for model.evaluate() later
test_ds = test_full_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Bayesian hyperparameter search. kernel size, first filter count, and learning rate
# stay fixed at values already justified by the manual tuning from before, only the later filters/dense size/dropout are searched
def build_model(hp):
    filters_2 = hp.Choice('filters_2', [128])
    filters_3 = hp.Choice('filters_3', [256])
    dense_units = hp.Choice('dense_units', [128])
    dropout_rate = hp.Float('dropout_rate', 0.30000000000000004, 0.4, step=0.1)

    learning_rate = hp.Choice('learning_rate', [1e-4, 3e-4, 5e-4, 1e-3])

    model = Sequential([

        tf.keras.Input(shape=input_shape),

        # Block 1
        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(32, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),

        # Block 2
        Conv2D(filters_2, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(filters_2, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),

        # Block 3
        Conv2D(filters_3, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(filters_3, (3, 3), padding="same"),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),
        GlobalAveragePooling2D(),
        Dense(dense_units),
        Activation('relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')

    ])

    model.compile(loss='binary_crossentropy',
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),  # Higher initial LR which anneals as val loss plateaus
                  metrics=['accuracy'])
    return model

tuner = kt.BayesianOptimization(
    build_model,
    objective='val_loss',
    max_trials=15,
    directory='tuner_results',
    project_name='deepfake_cnn_tuning',
    overwrite=True
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=tuning_epochs,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
    ]
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print("Best hyperparameters:", best_hp.values)

# k-fold validation of the best config only not every trial, saves some time
k_folds = 3

# Data aug only in training
cv_pool_ds = (train_full_ds
              .take(train_sample_size // batch_size)
              .concatenate(val_full_ds.take(validation_sample_size // batch_size))
              .map(normalize, num_parallel_calls=AUTOTUNE)
              .unbatch())

fold_val_losses = []
fold_val_accuracies = []

for fold in range(k_folds):
    fold_val_ds = (cv_pool_ds.enumerate()
                   .filter(lambda i, data: i % k_folds == fold)
                   .map(lambda i, data: data)
                   .batch(batch_size)
                   .prefetch(AUTOTUNE))

    fold_train_ds = (cv_pool_ds.enumerate()
                      .filter(lambda i, data: i % k_folds != fold)
                      .map(lambda i, data: data)
                      .batch(batch_size)
                      .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)

                      .prefetch(AUTOTUNE))

    fold_model = tuner.hypermodel.build(best_hp)
    fold_model.fit(fold_train_ds, validation_data=fold_val_ds, epochs=validation_epochs,
                    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
                    verbose=0)

    val_loss, val_acc = fold_model.evaluate(fold_val_ds, verbose=0)
    fold_val_losses.append(val_loss)
    fold_val_accuracies.append(val_acc)
    print(f"Fold {fold+1}/{k_folds}: val_loss={val_loss:.4f}, val_accuracy={val_acc:.4f}")

print(f"\nK-Fold CV results for best hyperparameters:")
print(f"Mean val_loss: {np.mean(fold_val_losses):.4f} (+/- {np.std(fold_val_losses):.4f})")
print(f"Mean val_accuracy: {np.mean(fold_val_accuracies):.4f} (+/- {np.std(fold_val_accuracies):.4f})")

Trial 4 Complete [00h 09m 14s]
val_loss: 0.4334363341331482

Best val_loss So Far: 0.3035700023174286
Total elapsed time: 00h 36m 56s
Best hyperparameters: {'filters_2': 128, 'filters_3': 256, 'dense_units': 128, 'dropout_rate': 0.30000000000000004, 'learning_rate': 0.001}


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


Fold 1/3: val_loss=0.3347, val_accuracy=0.8511
Fold 2/3: val_loss=0.3454, val_accuracy=0.8442
Fold 3/3: val_loss=0.2686, val_accuracy=0.8841

K-Fold CV results for best hyperparameters:
Mean val_loss: 0.3162 (+/- 0.0339)
Mean val_accuracy: 0.8598 (+/- 0.0174)


In [12]:
best_epoch = np.argmin(history.history['val_loss'])

print(f"Best Epoch: {best_epoch + 1}")
print(f"Best Training Loss: {history.history['loss'][best_epoch]}")
print(f"Best Validation Loss: {history.history['val_loss'][best_epoch]}")
print(f"Best Training Accuracy: {history.history['accuracy'][best_epoch]}")
print(f"Best Validation Accuracy: {history.history['val_accuracy'][best_epoch]}")


NameError: name 'history' is not defined